# GTA 6 Clip Factory — Colab Edition

Colab-first workflow for your existing GTA6_CLIPPING Drive project. This notebook reuses your completed transcript and candidate CSV; it does **not** re-transcribe the 49-minute source or require a GPU for review.

Current phase: validate source + candidate timestamps, create contact sheets, and render review previews at the source aspect ratio. No automatic publishing and no claim that transcript scores predict virality.

Run top to bottom. Keep your source video and working files in Google Drive.

In [ ]:
import os, sys, subprocess, pathlib, json
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/martyroy6996-spec/remotion-shorts-engine.git'
REPO_BRANCH = 'feature/gta6-colab-pipeline'
REPO_DIR = pathlib.Path('/content/remotion-shorts-engine')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pillow'], check=True)
print('FFmpeg:', subprocess.check_output(['ffmpeg', '-version'], text=True).splitlines()[0])
print('Repository:', REPO_DIR)

print('Running pipeline unit tests...')
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=REPO_DIR, check=True)


## 1. Point the notebook at the existing Drive project

Do not move or re-upload the 1.37 GB source. The paths below match the folders already created in your Drive.

In [ ]:
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/GTA6_CLIPPING')
VIDEO = ROOT / '02_SOURCE_MEDIA/authorized_originals/GTA-6-Gameplay-SnapYT.App.mp4'
TRANSCRIPTS = ROOT / '04_TRANSCRIPTS'
CANDIDATES = ROOT / '05_CLIP_CANDIDATES/ranked_clip_candidates.csv'
REVIEW_DIR = ROOT / '05_CLIP_CANDIDATES/visual_review_engine'
PREVIEW_DIR = ROOT / '05_CLIP_CANDIDATES/preview_videos_engine'
assert ROOT.exists(), f'Drive project folder missing: {ROOT}'
assert VIDEO.is_file(), f'Source video missing: {VIDEO}'
assert CANDIDATES.is_file(), f'Candidate CSV missing: {CANDIDATES}'
print('Source:', VIDEO.name, f'({VIDEO.stat().st_size / 1e9:.2f} GB)')
print('Existing transcript files:', [p.name for p in TRANSCRIPTS.glob('gta6_full_transcript.*')])
print('Candidate CSV:', CANDIDATES)


## 2. Validate source and candidate timecodes

This catches missing files and invalid timestamps before any expensive rendering.

In [ ]:
import subprocess
duration = float(subprocess.check_output(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(VIDEO)], text=True).strip())
print(f'Video duration: {duration:.2f}s ({duration/60:.2f} minutes)')
sys.path.insert(0, str(REPO_DIR / 'pipeline'))
from gta6_review import load_candidates
candidate_rows = load_candidates(CANDIDATES, 30)
for i, c in enumerate(candidate_rows, 1):
    assert 0 <= c['start'] < c['end'] <= duration + 1, f'Invalid candidate #{i}: {c}'
print('Valid candidates:', len(candidate_rows))
for i, c in enumerate(candidate_rows[:10], 1):
    print(f"{i:02d} | {int(c['start']//60):02d}:{int(c['start']%60):02d} | score {c['score']} | {c['transcript'][:180]}")


## 3. Extract visual contact sheets

Four frames per candidate at +5, +15, +25, and +35 seconds. Existing outputs are not overwritten; the review assets are written to a separate folder.

In [ ]:
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, str(REPO_DIR/'pipeline/gta6_review.py'), 'inspect', '--video', str(VIDEO), '--candidates', str(CANDIDATES), '--limit', '20', '--output', str(REVIEW_DIR)], check=True)
print('Contact sheets:', sorted(p.name for p in REVIEW_DIR.glob('visual_contact_sheet_*.jpg')))


## 4. Render source-aspect review previews

These are review files, not final vertical edits. They preserve the original gameplay aspect ratio so important action is not cut off by a forced centre crop.

In [ ]:
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, str(REPO_DIR/'pipeline/gta6_review.py'), 'previews', '--video', str(VIDEO), '--candidates', str(CANDIDATES), '--limit', '5', '--seconds', '45', '--output', str(PREVIEW_DIR)], check=True)
print('Review previews:', sorted(p.name for p in PREVIEW_DIR.glob('*.mp4')))


## 5. Inspect and approve manually

Open `GTA6_CLIPPING/05_CLIP_CANDIDATES/visual_review_engine` and `preview_videos_engine` in Drive. Only after watching complete previews should a candidate be approved for a vertical edit.

**Important:** the current review stage is implemented here. Final vertical composition, subtitle correction, audio polish, and Remotion/HyperFrames rendering are separate production stages and must be validated before being treated as finished.